# Drought Severity Prediction (Arthur v8)
**Schnellere Version für Kaggle**
* `sample_submission` Abhängigkeit entfernt (baut die CSV direkt)
* `stride=2` und `N_ESTIMATORS=800` für schnelleres Training (ca. 60-90 Min statt 150 Min)
* Two-Stage Hurdle Model für Zero-Inflation


In [ ]:
import time
import warnings
from pathlib import Path
import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb

try:
    from catboost import CatBoostRegressor
    CATBOOST_AVAILABLE = True
except ImportError:
    CATBOOST_AVAILABLE = False

warnings.filterwarnings("ignore")

# ─── Pfade ────────────────────────────────────────────────────────────────────
DATA_DIR = Path("/kaggle/input/datasets/axxtur/data-mining-2026-final-assignment")
if not DATA_DIR.exists(): # Fallback if standard Kaggle path is different
    DATA_DIR = Path("/kaggle/input/data-mining-2026-final-assignment")
    if not DATA_DIR.exists():
        DATA_DIR = Path("data") # Local testing fallback

OUT_PATH = Path("/kaggle/working/submission_arthur_v8.csv")
if not Path("/kaggle/working").exists():
    OUT_PATH = Path("submission_arthur_v8.csv")

TRAIN_CSV = DATA_DIR / "train.csv"
TEST_CSV  = DATA_DIR / "test.csv"

# ─── Beschleunigte Parameter ──────────────────────────────────────────────────
# stride=2 halbiert die Trainingsdaten (fast gleiche Performance, 2x schneller)
WINDOW_STRIDE = 2
N_ESTIMATORS  = 800  # Weniger Trees, Early Stopping fängt es eh meist früher

RANDOM_STATE    = 42
VAL_REGION_FRAC = 0.20
WEEK_BUCKET     = 7
DRY_THRESHOLD   = 1.0

# ─── Features ─────────────────────────────────────────────────────────────────
WEATHER_COLS = [
    "prec", "surf_pre", "humidity", "tmp", "dp_tmp", "wb_tmp",
    "tmp_max", "tmp_min", "tmp_range", "surf_tmp",
    "wind", "wind_max", "wind_min", "wind_range",
]
LAG_COLS  = ["tmp_range", "tmp_max", "tmp", "prec", "wind", "surf_pre", "humidity"]
LAGS      = [1, 3, 7, 14, 21]
ROLL_COLS = ["prec", "wind", "tmp", "humidity", "surf_pre"]
ROLL_WINS = [7, 14, 30, 60, 90, 180]

def build_feature_list() -> list[str]:
    lag_names  = [f"{c}_lag{lag}" for c in LAG_COLS for lag in LAGS]
    roll_names = [f"{col}_roll{w}_{stat}" for col in ROLL_COLS for w in ROLL_WINS for stat in ("mean", "std", "max")]
    calendar = ["month_sin", "month_cos", "day_sin", "day_cos", "week_sin", "week_cos"]
    drought_indices = ["prec_deficit_90d", "prec_trend_30d", "humidity_deficit_90d",
                       "tmp_anomaly_90d", "heat_drought_idx", "dry_days_14d", "dry_days_30d"]
    extra = ["regional_mean_score"]
    return WEATHER_COLS + lag_names + roll_names + calendar + drought_indices + extra

NUM_FEATURES = build_feature_list()

# ─── Model Config ─────────────────────────────────────────────────────────────
LGB_PARAMS = dict(objective="regression", metric="mae", n_estimators=N_ESTIMATORS, learning_rate=0.04,
                  num_leaves=127, min_child_samples=60, subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, verbose=-1)

XGB_PARAMS = dict(objective="reg:squarederror", n_estimators=N_ESTIMATORS, learning_rate=0.04,
                  max_depth=6, min_child_weight=50, subsample=0.8, colsample_bytree=0.8,
                  reg_alpha=0.1, reg_lambda=1.0, tree_method="hist", n_jobs=-1, verbosity=0)

CAT_PARAMS = dict(iterations=N_ESTIMATORS, learning_rate=0.04, depth=6, loss_function="MAE",
                  eval_metric="MAE", random_seed=RANDOM_STATE, verbose=False, thread_count=-1)

CLF_PARAMS = dict(objective="binary", metric="binary_logloss", n_estimators=N_ESTIMATORS,
                  learning_rate=0.04, num_leaves=63, min_child_samples=100, subsample=0.8,
                  colsample_bytree=0.8, reg_alpha=0.1, reg_lambda=0.1, n_jobs=-1, verbose=-1)


In [ ]:
# ─── Hilfsfunktionen ───────────────────────────────────────────────────────────
def _parse_dates_inplace(df: pd.DataFrame) -> None:
    parts = df["date"].str.split("-", expand=True)
    df["year"]  = parts[0].astype(np.int32)
    df["month"] = parts[1].astype(np.int32)
    df["day"]   = parts[2].astype(np.int32)
    df["ordinal"] = df["year"] * 372 + df["month"] * 31 + df["day"]

def elapsed(t0: float) -> str:
    s = time.time() - t0
    return f"{s/60:.1f} Min." if s >= 60 else f"{s:.0f}s"

def mae(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return float(np.mean(np.abs(np.clip(y_pred, 0, 5) - y_true)))

def show_mae(name: str, y_true: np.ndarray, y_pred: np.ndarray) -> None:
    print(f"  {name:<52s}  MAE = {mae(y_true, y_pred):.4f}")

def compute_regional_mean_score(train_raw: pd.DataFrame) -> pd.Series:
    return train_raw.groupby("region_id")["score"].mean()

def add_regional_mean_score(df: pd.DataFrame, region_means: pd.Series) -> pd.DataFrame:
    df["regional_mean_score"] = df["region_id"].map(region_means).astype(np.float32)
    return df

# ─── Feature Engineering ──────────────────────────────────────────────────────
def compute_region_features(tr: pd.DataFrame, te: pd.DataFrame) -> tuple[pd.DataFrame, pd.DataFrame]:
    te = te.copy()
    te["score"] = np.nan
    panel = pd.concat([tr, te], ignore_index=True).sort_values("ordinal").reset_index(drop=True)
    new_cols: dict[str, np.ndarray] = {}

    new_cols["month_sin"] = np.sin(2 * np.pi * panel["month"] / 12).astype(np.float32)
    new_cols["month_cos"] = np.cos(2 * np.pi * panel["month"] / 12).astype(np.float32)
    new_cols["day_sin"]   = np.sin(2 * np.pi * panel["day"]   / 31).astype(np.float32)
    new_cols["day_cos"]   = np.cos(2 * np.pi * panel["day"]   / 31).astype(np.float32)

    week_of_year = (panel["ordinal"] // 7) % 52
    new_cols["week_sin"] = np.sin(2 * np.pi * week_of_year / 52).astype(np.float32)
    new_cols["week_cos"] = np.cos(2 * np.pi * week_of_year / 52).astype(np.float32)

    for col in LAG_COLS:
        s = panel[col]
        for lag in LAGS:
            new_cols[f"{col}_lag{lag}"] = s.shift(lag).astype(np.float32)

    for col in ROLL_COLS:
        prior = panel[col].shift(1)
        for w in ROLL_WINS:
            min_p = max(3, w // 10)
            r = prior.rolling(w, min_periods=min_p)
            new_cols[f"{col}_roll{w}_mean"] = r.mean().astype(np.float32)
            new_cols[f"{col}_roll{w}_std"]  = r.std().astype(np.float32)
            new_cols[f"{col}_roll{w}_max"]  = r.max().astype(np.float32)

    prec_prior = panel["prec"].shift(1)
    new_cols["prec_deficit_90d"] = (prec_prior.rolling(90, min_periods=30).mean() - prec_prior.rolling(365, min_periods=60).mean()).astype(np.float32)
    p7   = prec_prior.rolling(7,  min_periods=3).mean()
    p30  = prec_prior.rolling(30, min_periods=10).mean()
    p30s = prec_prior.rolling(30, min_periods=10).std().clip(lower=0.01)
    new_cols["prec_trend_30d"] = ((p7 - p30) / p30s).astype(np.float32)

    hum_prior = panel["humidity"].shift(1)
    new_cols["humidity_deficit_90d"] = (hum_prior.rolling(90, min_periods=30).mean() - hum_prior.rolling(365, min_periods=60).mean()).astype(np.float32)

    tmp_prior   = panel["tmp"].shift(1)
    tmp_anomaly = (tmp_prior.rolling(90,  min_periods=30).mean() - tmp_prior.rolling(365, min_periods=60).mean()).astype(np.float32)
    new_cols["tmp_anomaly_90d"] = tmp_anomaly

    new_cols["heat_drought_idx"] = (new_cols["prec_deficit_90d"] * tmp_anomaly.clip(lower=0)).astype(np.float32)

    dry = (panel["prec"].shift(1) < DRY_THRESHOLD).astype(np.float32)
    new_cols["dry_days_14d"] = dry.rolling(14, min_periods=3).sum().astype(np.float32)
    new_cols["dry_days_30d"] = dry.rolling(30, min_periods=7).sum().astype(np.float32)

    panel = pd.concat([panel, pd.DataFrame(new_cols, index=panel.index)], axis=1)
    n_tr = len(tr)
    return panel.iloc[:n_tr].copy(), panel.iloc[n_tr:].copy()


In [ ]:
# ─── Dataset Assembly ─────────────────────────────────────────────────────────
def daily_to_weekly(df: pd.DataFrame) -> pd.DataFrame:
    week = df["ordinal"] // WEEK_BUCKET
    idx  = df.groupby(week, sort=False)["ordinal"].idxmax()
    return df.loc[idx].reset_index(drop=True)

def build_sliding_windows(weekly: pd.DataFrame, skip_regions: set, num_features: list[str], stride: int = 1) -> tuple[pd.DataFrame, np.ndarray]:
    X_parts, y_parts, region_parts = [], [], []
    for region, g in weekly.groupby("region_id", sort=False):
        if region in skip_regions: continue
        g = g.sort_values("ordinal")
        scores = g["score"].to_numpy(dtype=np.float32)
        X_num  = g[num_features].to_numpy(dtype=np.float32)
        n = len(g)
        if n < 6: continue
        n_win = n - 5
        y_reg = np.lib.stride_tricks.sliding_window_view(scores[1:], 5)[:n_win]
        idx   = list(range(0, n_win, stride))
        if (n_win - 1) not in idx: idx.append(n_win - 1)
        X_parts.append(X_num[idx])
        y_parts.append(y_reg[idx])
        region_parts.extend([region] * len(idx))
    X_df = pd.DataFrame(np.vstack(X_parts).astype(np.float32), columns=num_features)
    X_df["region_id"] = pd.Categorical(region_parts)
    return X_df, np.vstack(y_parts).astype(np.float32)

def build_val_samples(weekly: pd.DataFrame, val_regions: list, num_features: list[str]) -> tuple[pd.DataFrame, np.ndarray]:
    X_parts, y_parts, r_parts = [], [], []
    for region in val_regions:
        g = weekly.loc[weekly["region_id"] == region].sort_values("ordinal")
        if len(g) < 6: continue
        X_parts.append(g.iloc[-6][num_features].to_numpy(dtype=np.float32))
        y_parts.append(g.iloc[-5:]["score"].to_numpy(dtype=np.float32))
        r_parts.append(region)
    X_df = pd.DataFrame(np.vstack(X_parts), columns=num_features)
    X_df["region_id"] = pd.Categorical(r_parts)
    return X_df, np.vstack(y_parts)


In [ ]:
# ─── Training Functions ───────────────────────────────────────────────────────
def train_lgb_models(X_tr, y_tr, X_va, y_va, n_trees_per_week=None):
    models = []
    for week in range(5):
        n = (n_trees_per_week[week] if n_trees_per_week else None) or LGB_PARAMS["n_estimators"]
        p = dict(LGB_PARAMS, random_state=RANDOM_STATE + week, n_estimators=n)
        m = lgb.LGBMRegressor(**p)
        fit_kw = dict(categorical_feature=["region_id"])
        if X_va is not None:
            fit_kw["eval_set"] = [(X_va, y_va[:, week].ravel())]
            fit_kw["eval_metric"] = "mae"
            fit_kw["callbacks"] = [lgb.early_stopping(50, verbose=False)]
        m.fit(X_tr, y_tr[:, week].ravel(), **fit_kw)
        models.append(m)
    return models

def train_xgb_models(X_tr, y_tr, X_va, y_va, num_features, n_trees_per_week=None):
    X_tr_n = X_tr[num_features].to_numpy(dtype=np.float32)
    X_va_n = X_va[num_features].to_numpy(dtype=np.float32) if X_va is not None else None
    models = []
    for week in range(5):
        n = (n_trees_per_week[week] if n_trees_per_week else None) or XGB_PARAMS["n_estimators"]
        p = dict(XGB_PARAMS, random_state=RANDOM_STATE + week, n_estimators=n)
        fit_kw = {}
        if X_va_n is not None:
            p["early_stopping_rounds"] = 50
            fit_kw["eval_set"] = [(X_va_n, y_va[:, week].ravel())]
            fit_kw["verbose"] = False
        m = xgb.XGBRegressor(**p)
        m.fit(X_tr_n, y_tr[:, week].ravel(), **fit_kw)
        models.append(m)
    return models

def train_cat_models(X_tr, y_tr, X_va, y_va, num_features, n_trees_per_week=None):
    if not CATBOOST_AVAILABLE: return None
    X_tr_n = X_tr[num_features].to_numpy(dtype=np.float32)
    X_va_n = X_va[num_features].to_numpy(dtype=np.float32) if X_va is not None else None
    models = []
    for week in range(5):
        n = (n_trees_per_week[week] if n_trees_per_week else None) or CAT_PARAMS["iterations"]
        p = dict(CAT_PARAMS, iterations=n, random_seed=RANDOM_STATE + week)
        fit_kw = {}
        if X_va_n is not None:
            fit_kw["eval_set"] = (X_va_n, y_va[:, week].ravel())
            fit_kw["early_stopping_rounds"] = 50
        m = CatBoostRegressor(**p)
        m.fit(X_tr_n, y_tr[:, week].ravel(), **fit_kw)
        models.append(m)
    return models

def train_lgb_classifiers(X_tr, y_tr, X_va, y_va, n_trees_per_week=None):
    models = []
    for week in range(5):
        y_bin = (y_tr[:, week] > 0).astype(np.int32)
        n = (n_trees_per_week[week] if n_trees_per_week else None) or CLF_PARAMS["n_estimators"]
        p = dict(CLF_PARAMS, random_state=RANDOM_STATE + week + 100, n_estimators=n)
        m = lgb.LGBMClassifier(**p)
        fit_kw = dict(categorical_feature=["region_id"])
        if X_va is not None:
            y_va_bin = (y_va[:, week] > 0).astype(np.int32)
            fit_kw["eval_set"] = [(X_va, y_va_bin)]
            fit_kw["eval_metric"] = "binary_logloss"
            fit_kw["callbacks"] = [lgb.early_stopping(50, verbose=False)]
        m.fit(X_tr, y_bin, **fit_kw)
        models.append(m)
    return models

def predict_lgb(models, X): return np.clip(np.column_stack([m.predict(X) for m in models]), 0.0, 5.0).astype(np.float32)
def predict_xgb(models, X, num_features): return np.clip(np.column_stack([m.predict(X[num_features].to_numpy(dtype=np.float32)) for m in models]), 0.0, 5.0).astype(np.float32)
def predict_cat(models, X, num_features): 
    if models is None: return None
    return np.clip(np.column_stack([m.predict(X[num_features].to_numpy(dtype=np.float32)) for m in models]), 0.0, 5.0).astype(np.float32)
def predict_clf_proba(models, X): return np.column_stack([m.predict_proba(X)[:, 1] for m in models]).astype(np.float32)

def _cat_best_iter(m, default):
    try: return int(m.get_best_iteration() or default)
    except: return default

def optimize_blend(y_va, lgb_val, xgb_val, cat_val=None):
    alphas = [0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9]
    best_mae_v, best_weights = 999.0, (0.5, 0.5, 0.0)
    if cat_val is not None:
        for a in alphas:
            for b in alphas:
                c = round(1.0 - a - b, 8)
                if c < 0: continue
                m = mae(y_va, a * lgb_val + b * xgb_val + c * cat_val)
                if m < best_mae_v: best_mae_v, best_weights = m, (a, b, c)
    else:
        for a in alphas:
            m = mae(y_va, a * lgb_val + (1 - a) * xgb_val)
            if m < best_mae_v: best_mae_v, best_weights = m, (a, 1 - a, 0.0)
    return best_weights, best_mae_v

def optimize_hurdle_alpha(y_va, blend_val, clf_proba_val):
    best_alpha, best_mae_v = 0.0, mae(y_va, blend_val)
    for alpha in np.arange(0.05, 1.01, 0.05):
        m = mae(y_va, np.clip((alpha * clf_proba_val + (1.0 - alpha)) * blend_val, 0.0, 5.0))
        if m < best_mae_v: best_alpha, best_mae_v = float(alpha), m
    return best_alpha, best_mae_v


In [ ]:
# ─── Hauptausführung ──────────────────────────────────────────────────────────
t0 = time.time()
print(f"Lade Daten aus {DATA_DIR} ...")
dtypes = {c: np.float32 for c in WEATHER_COLS}
train_raw = pd.read_csv(TRAIN_CSV, dtype=dtypes)
test_raw  = pd.read_csv(TEST_CSV,  dtype=dtypes)
_parse_dates_inplace(train_raw)
_parse_dates_inplace(test_raw)
train_raw["score"] = pd.to_numeric(train_raw["score"], errors="coerce").astype(np.float32)
regions = train_raw["region_id"].unique()
print(f"Train: {len(train_raw):,} | Test: {len(test_raw):,} | Regionen: {len(regions)}")

region_means = compute_regional_mean_score(train_raw)

print(f"\nFeature Engineering (Start: {elapsed(t0)}) ...")
train_by_region = {r: g.reset_index(drop=True) for r, g in train_raw.groupby("region_id", sort=False)}
test_by_region  = {r: g.reset_index(drop=True) for r, g in test_raw.groupby("region_id",  sort=False)}
del train_raw, test_raw

all_tr_feat, all_te_feat = [], []
for r in regions:
    tr_f, te_f = compute_region_features(train_by_region[r], test_by_region.get(r, pd.DataFrame()))
    all_tr_feat.append(tr_f)
    all_te_feat.append(te_f)

train_feat = add_regional_mean_score(pd.concat(all_tr_feat, ignore_index=True), region_means)
test_feat  = add_regional_mean_score(pd.concat(all_te_feat, ignore_index=True), region_means)
del all_tr_feat, all_te_feat
print(f"Wöchentliche Aggregation (Start: {elapsed(t0)}) ...")
train_weekly = pd.concat([daily_to_weekly(g) for _, g in train_feat[train_feat["score"].notna()].groupby("region_id", sort=False)], ignore_index=True)

print(f"\nSplitting & Windowing (Start: {elapsed(t0)}) ...")
rng = np.random.default_rng(RANDOM_STATE)
val_regions = set(rng.choice(regions, size=int(len(regions) * VAL_REGION_FRAC), replace=False))
X_tr, y_tr = build_sliding_windows(train_weekly, val_regions, NUM_FEATURES, stride=WINDOW_STRIDE)
X_va, y_va = build_val_samples(train_weekly, sorted(val_regions), NUM_FEATURES)
print(f"Training Windows: {len(X_tr):,} (stride={WINDOW_STRIDE})")

print(f"\nTrainiere Modelle (Start: {elapsed(t0)}) ...")
lgb_models = train_lgb_models(X_tr, y_tr, X_va, y_va)
lgb_val = predict_lgb(lgb_models, X_va)
xgb_models = train_xgb_models(X_tr, y_tr, X_va, y_va, NUM_FEATURES)
xgb_val = predict_xgb(xgb_models, X_va, NUM_FEATURES)

cat_val, cat_models_val = None, None
if CATBOOST_AVAILABLE:
    cat_models_val = train_cat_models(X_tr, y_tr, X_va, y_va, NUM_FEATURES)
    cat_val = predict_cat(cat_models_val, X_va, NUM_FEATURES)

(lgb_w, xgb_w, cat_w), best_mae_val = optimize_blend(y_va, lgb_val, xgb_val, cat_val)
blend_val = lgb_w * lgb_val + xgb_w * xgb_val + (cat_w * cat_val if cat_val is not None else 0)
print(f"Base Ensemble MAE: {best_mae_val:.4f} (LGB={lgb_w:.2f}, XGB={xgb_w:.2f}, CAT={cat_w:.2f})")

print(f"\nTrainiere Two-Stage Hurdle Classifiers ...")
clf_models = train_lgb_classifiers(X_tr, y_tr, X_va, y_va)
best_alpha, best_hurdle_mae = optimize_hurdle_alpha(y_va, blend_val, predict_clf_proba(clf_models, X_va))
print(f"Hurdle MAE: {best_hurdle_mae:.4f} (Alpha={best_alpha:.2f}) -> Verbesserung: -{best_mae_val - best_hurdle_mae:.4f}")

print(f"\nFinales Retraining auf allen Daten (Start: {elapsed(t0)}) ...")
X_all, y_all = build_sliding_windows(train_weekly, set(), NUM_FEATURES, stride=WINDOW_STRIDE)

final_lgb = train_lgb_models(X_all, y_all, None, None, [int(getattr(m, "best_iteration_", None) or LGB_PARAMS["n_estimators"]) for m in lgb_models])
final_xgb = train_xgb_models(X_all, y_all, None, None, NUM_FEATURES, [int(getattr(m, "best_iteration", None) or XGB_PARAMS["n_estimators"]) for m in xgb_models])
final_cat = train_cat_models(X_all, y_all, None, None, NUM_FEATURES, [_cat_best_iter(m, CAT_PARAMS["iterations"]) for m in cat_models_val]) if cat_models_val else None
final_clf = train_lgb_classifiers(X_all, y_all, None, None, [int(getattr(m, "best_iteration_", None) or CLF_PARAMS["n_estimators"]) for m in clf_models]) if best_alpha > 0 else None

print(f"\nErzeuge Submission ohne externe sample_submission.csv Abhängigkeit ...")
X_test = test_feat.sort_values(["region_id", "ordinal"]).groupby("region_id", sort=False).tail(1)[["region_id"] + NUM_FEATURES].reset_index(drop=True)
X_test["region_id"] = X_test["region_id"].astype("category")

test_preds = lgb_w * predict_lgb(final_lgb, X_test) + xgb_w * predict_xgb(final_xgb, X_test, NUM_FEATURES)
if final_cat: test_preds += cat_w * predict_cat(final_cat, X_test, NUM_FEATURES)
if final_clf and best_alpha > 0:
    # test_preds = np.clip((best_alpha * predict_clf_proba(final_clf, X_test) + (1.0 - best_alpha)) * test_preds, 0.0, 5.0) # DISABLED
    pass # Hurdle disabled for final submission due to Kaggle Distribution Shift

# BAUE SUBMISSION DIREKT AUS X_TEST (100% BULLETPROOF)
sub = pd.DataFrame({"region_id": X_test["region_id"].values})
for k in range(5):
    sub[f"pred_week{k+1}"] = test_preds[:, k]

# FIX ROW ORDER: Sort numerically by region_id (R1, R2, ..., R10) instead of lexicographically (R1, R10, R100)
sub["region_num"] = sub["region_id"].astype(str).str.extract(r"(\d+)").astype(int)
sub = sub.sort_values("region_num").drop(columns=["region_num"])

# Stelle sicher, dass die Spalten exakt stimmen (kein Merge mit sample_submission nötig)
sub.to_csv(OUT_PATH, index=False)
print(f"✅ Submission gespeichert unter: {OUT_PATH}")
print(f"Gesamtlaufzeit: {elapsed(t0)}")
